In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [11]:
df=pd.read_csv("../data/creditcard.csv")
df.head()
df = df.drop_duplicates()
print(len(df))   # should print 283726

283726


In [12]:
# sort by time (you already ran this, it's fine to keep)
df = df.sort_values("Time").reset_index(drop=True)

# cut points
n = len(df)                  # total number of rows
train_end = int(n * 0.6)     # row number where training stops
val_end = int(n * 0.8)       # row number where validation stops

# the three parts
train = df.iloc[0:train_end]          # from row 0 up to train_end
val   = df.iloc[train_end:val_end]    # from train_end up to val_end
test  = df.iloc[val_end:]             # from val_end to the very end


In [13]:
print("train:", len(train), "rows,", train["Class"].sum(), "frauds")
print("val:", len(val), "rows,", val["Class"].sum(), "frauds")
print("test:", len(test), "rows,", test["Class"].sum(), "frauds")

train: 170235 rows, 342 frauds
val: 56745 rows, 57 frauds
test: 56746 rows, 74 frauds


In [22]:
def add_tax(price):
    total = price * 1.13
    return total

add_tax(100)
add_tax(50)
print(round(add_tax(100),2))
print(round(add_tax(50),2))

113.0
56.5


In [23]:
#define total cost
def total_cost(is_fraud, flagged, amount, review_cost=2.5):
    #every flagged transaction is reviewd
    reviews = flagged.sum()*review_cost
    #frauds that were not flagged : we lose their amount
    missed = amount [(is_fraud ==1)&(flagged==0)].sum()
    return reviews +missed

In [24]:
# Strategy A "flag nothing": no alerts at all, so every fraud is missed
y = val["Class"]
amt = val["Amount"]
flag_nothing = pd.Series(False, index=val.index)
cost_nothing = total_cost(y, flag_nothing, amt)
print("Flag nothing:", round(cost_nothing, 2))

Flag nothing: 12231.09


In [25]:
# Strategy B "flag all":
y = val["Class"]
amt = val["Amount"]
flag_all = pd.Series(True, index=val.index)
cost_all = total_cost(y, flag_all, amt)
print("Flag all:", round(cost_all, 2))

Flag all: 141862.5


In [26]:
# Strategy C "flag amounts over €500": only big transactions get reviewed
y = val["Class"]
amt = val["Amount"]
flag_big = amt>500
cost_big = total_cost(y, flag_big, amt)
print("Flag big:", round(cost_big, 2))

Flag big: 9209.33


## Baseline results (validation set, review cost €2.50)

| Strategy | Cost |
|---|---|
| Flag nothing | €12,231.09 |
| Flag all | €141,862.50 |
| Flag amount > €500 | €9,209.33 |

**Takeaways:**
- Flagging everything is the worst option: fraud is so rare that review costs (56,745 × €2.50) far outweigh the fraud caught.
- A simple "amount > €500" rule saves about €3,000 (25%) compared with doing nothing, but it misses most frauds, since half of all frauds are under €10 and only a handful are above €500.
- The cost to beat for any model is **€9,209.33**.

Review cost of €2.50 per alert follows Correa Bahnsen et al. (2013), "Cost Sensitive Credit Card Fraud Detection using Bayes Minimum Risk".